# Exploring speech-emotion data

Run `python download_data.py` first. This notebook shows **what the CNN actually sees**.

## What is a spectrogram?
- A **waveform** is loudness over time. Pitch and timbre are hard to read from it.
- Cut the audio into tiny overlapping windows and run a Fourier transform on each. This tells you *how much of each frequency* is present in that instant.
- Stack the windows side by side and you get an **image**: x = time, y = frequency, brightness = energy.
- The **Mel** scale squeezes the frequency axis to match human hearing. The **log** (decibel) scale matches how we perceive loudness.

The result is a 128 × 130 image that a CNN can read like any other picture.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # so `import src` works from notebooks/

import numpy as np, librosa, librosa.display, matplotlib.pyplot as plt, IPython.display as ipd
from src import config as C
from src.features import load_audio, preprocess_waveform, log_mel, normalize, list_clips
from src.augment import add_noise, pitch_shift, spec_augment

C.set_seed()
rng = np.random.default_rng(C.SEED)
classes, clips = list_clips()
print(classes, len(clips), "clips")

## 1. Class balance
RAVDESS is balanced by design (each actor records every emotion equally often). 'Neutral' has half as many clips if you include it.

In [ ]:
counts = [sum(l == i for _, l in clips) for i in range(len(classes))]
plt.bar(classes, counts); plt.ylabel("clips"); plt.title("Clips per emotion"); plt.show()

## 2. One clip: waveform → spectrogram → Mel → log-Mel
Same sound, four views. The last one is what the model gets (before normalisation).

In [ ]:
path = next(p for p, l in clips if classes[l] == "angry")
y = preprocess_waveform(load_audio(path))
ipd.display(ipd.Audio(y, rate=C.SAMPLE_RATE))

linear_db = librosa.amplitude_to_db(np.abs(librosa.stft(y, n_fft=C.N_FFT, hop_length=C.HOP_LENGTH)), ref=np.max)
fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))
librosa.display.waveshow(y, sr=C.SAMPLE_RATE, ax=ax[0]); ax[0].set_title("Waveform")
librosa.display.specshow(linear_db, sr=C.SAMPLE_RATE, hop_length=C.HOP_LENGTH, x_axis="time", y_axis="hz", ax=ax[1]); ax[1].set_title("Linear spectrogram (dB)")
img = librosa.display.specshow(log_mel(y), sr=C.SAMPLE_RATE, hop_length=C.HOP_LENGTH, x_axis="time", y_axis="mel", ax=ax[2]); ax[2].set_title("Log-Mel spectrogram (dB)")
fig.colorbar(img, ax=ax[2]); plt.tight_layout(); plt.show()

## 3. Average spectrogram per emotion
Angry/happy speech tends to carry more high-frequency energy than sad speech. This is the pattern the CNN picks up on.

In [ ]:
X = np.load(C.PROCESSED_DIR / "X.npy") if (C.PROCESSED_DIR / "X.npy").exists() else None
if X is None:
    print("Run `python -m src.features` first.")
else:
    labels = np.load(C.PROCESSED_DIR / "y.npy")
    fig, ax = plt.subplots(1, len(classes), figsize=(4 * len(classes), 3.5), sharey=True)
    for i, name in enumerate(classes):
        ax[i].imshow(X[labels == i].mean(axis=0), origin="lower", aspect="auto", cmap="magma")
        ax[i].set_title(name); ax[i].set_xlabel("time frame")
    ax[0].set_ylabel("mel band"); plt.tight_layout(); plt.show()

## 4. Augmentation
Noise and pitch shift happen on the waveform. SpecAugment masks happen on the spectrogram (fresh every epoch).

In [ ]:
noisy = add_noise(y, rng); shifted = pitch_shift(y, rng)
spec = normalize(log_mel(y))
views = {"original": spec, "+ noise": normalize(log_mel(noisy)),
         "pitch shifted": normalize(log_mel(shifted)), "SpecAugment": spec_augment(spec, rng)}
fig, ax = plt.subplots(1, 4, figsize=(16, 3.2))
for a, (name, s) in zip(ax, views.items()):
    a.imshow(s, origin="lower", aspect="auto", cmap="magma"); a.set_title(name)
plt.tight_layout(); plt.show()

## 5. After training
`python train.py` writes plots and a report to `assets/`. Remember the caveat: these are **acted** emotions from 24 actors, so accuracy here overstates real-world performance (see README → Limitations).

In [ ]:
from IPython.display import Image, display
for f in ["training_curves.png", "confusion_matrix.png"]:
    p = C.ASSETS_DIR / f
    if p.exists(): display(Image(filename=p))
rp = C.ASSETS_DIR / "classification_report.txt"
print(rp.read_text() if rp.exists() else "Train first: python train.py")